# Project

In [1]:
import os
import time

from huggingface_hub import hf_hub_download

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import welch
from scipy.stats import skew, kurtosis

from sklearn.base import clone

from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.decomposition import PCA

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, f1_score, confusion_matrix

from concurrent.futures import ThreadPoolExecutor
from time import perf_counter

from sklearn.metrics import accuracy_score

/home/maxence/Desktop/university/fourth-year/hoai/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Preprocessing
### Data Loading

Preprocessed dataset from: [monster-monash/STEW](https://huggingface.co/datasets/monster-monash/STEW)

In [2]:
REPO_ID = "monster-monash/STEW"

def download_file(filename):
    return hf_hub_download(repo_id=REPO_ID,filename=filename,repo_type="dataset")

X_path = download_file("STEW_X.npy")

Y_path = download_file("STEW_y.csv")

SUBJECT_ID_path = download_file("STEW_subject_id.csv")

X = np.load(X_path)
Y = pd.read_csv(Y_path, header= None).squeeze().to_numpy()
SUBJECT_ID = pd.read_csv(SUBJECT_ID_path, header=None).squeeze().to_numpy()

### Data Exploration

In [3]:
# We verify the shapes of the three arrays

assert(X.ndim == 3)
assert(X.shape == (28512,14,256)) # Expected shape from the tabular dataset description
assert(len(X)==len(Y)==len(SUBJECT_ID))

We make plots to get a better sense of how the data is structured

In [4]:
EEG_CHANNELS = ["AF3", "F7", "F3", "FC5", "T7", "P7", "O1", "O2", "P8", "T8", "FC6", "F4", "F8", "AF4"]
SAMPLING_FREQUENCY = 128

We can see that the data is equally divided by workload classes and that the same amount of samples exists for each subject.

### Data Cleaning

We remove all the data from subjects 5, 24, 42, as the dataset presentation article (Lim et al, 2018) states that ratings were not obtained for this subjects.

In [5]:
SUBJECTS_TO_REMOVE=[5,24,42]

mask=np.logical_not(np.isin(SUBJECT_ID,SUBJECTS_TO_REMOVE))
X=X[mask]
Y=Y[mask]
SUBJECT_ID=SUBJECT_ID[mask]

In [6]:
#We test that the removal was successful
for subject in SUBJECTS_TO_REMOVE:
    assert(X[SUBJECT_ID==subject].size==0)
    assert(Y[SUBJECT_ID==subject].size==0)
    assert(SUBJECT_ID[SUBJECT_ID==subject].size==0)

    assert(X[SUBJECT_ID==(subject-1)].size!=0)
    assert(Y[SUBJECT_ID==(subject-1)].size!=0)
    assert(SUBJECT_ID[SUBJECT_ID==(subject-1)].size!=0)


assert(X.shape == (28512-28512*3/48,14,256)) # Expected new shape, given that we removed 3/48th of the data
assert(len(X)==len(Y)==len(SUBJECT_ID))

## Feature engineering

We crete a .csv table with the chosen features:

In [7]:
def extract_features(signal):

    # Time-domain features
    new_features = {
        "mean": np.mean(signal),
        "std" : np.std(signal),
        "var" : np.var(signal),
        "rms" : np.sqrt(np.mean(signal**2)),
        "skewness": skew(signal, bias=True),
        "kurtosis": kurtosis(signal, fisher=True, bias=True),
    }
    # Frequency-domain features
    freqs, psd = welch(
        signal,
        fs=SAMPLING_FREQUENCY,
        nperseg=len(signal)
    )

    frequency_resolution = freqs[1] - freqs[0] 

    total_mask = (freqs >= 1) & (freqs < 45)
    total_power = np.sum(psd[total_mask]) * frequency_resolution

    for band, (low, high) in FREQUENCY_BANDS.items():

        mask = (freqs >= low) & (freqs < high)

        power = (
            np.sum(psd[mask]) * frequency_resolution
            if np.any(mask) else 0.0
        )

        new_features[f"{band}_power"] = power

        new_features[f"{band}_relative_power"] = (
            power / total_power
            if total_power > 1e-12 else 0.0
        )

    # Spectral entropy over the 1–45 Hz range
    psd_band = psd[total_mask]
    power_sum = psd_band.sum()

    if power_sum > 1e-12:
        probabilities = psd_band / power_sum
        probabilities = probabilities[probabilities > 0]

        new_features["spectral_entropy"] = -np.sum(
            probabilities * np.log2(probabilities)
        )
    else:
        new_features["spectral_entropy"] = 0.0

    return new_features

In [8]:
def create_dataset():
    t0 = time.time()
    records = []

    for sample_idx in range(len(X)):

        eeg_sample = X[sample_idx]

        record = {
            "sample_id": sample_idx,
            "subject_id": SUBJECT_ID[sample_idx],
        }

        for channel_idx, channel_name in enumerate(EEG_CHANNELS):

            signal = eeg_sample[channel_idx, :]

            channel_features = extract_features(signal)

            for feature_name, value in channel_features.items():
                column_name = f"{channel_name}_{feature_name}"
                record[column_name] = value

        record["label"] = Y[sample_idx]
        records.append(record)

        if (sample_idx + 1) % 1000 == 0:
            print(f"Processed {sample_idx + 1}/{len(X)} examples")

    print(f"Processed {sample_idx + 1}/{len(X)} examples")

    print(f"Time required: {time.time() - t0}")
    return records

We create a dataframe with the extracted outputs and write it to a .csv file.

In [9]:
DATASET_PATH = "STEW_features.csv"

if(os.path.isfile(DATASET_PATH)):
    DATASET_EXT=pd.read_csv(DATASET_PATH)
else:
    new_dataset=create_dataset()
    DATASET_EXT = pd.DataFrame(new_dataset)
    DATASET_EXT.to_csv(DATASET_PATH, index=False)
    
print(DATASET_EXT.shape)

(26730, 241)


In [10]:
print(DATASET_EXT.head())
print(DATASET_EXT.shape)

   sample_id  subject_id  AF3_mean    AF3_std    AF3_var    AF3_rms  \
0          0          12 -2.364527  16.718996  279.52484  16.885374   
1          1          12  1.513650  15.534682  241.32635  15.608251   
2          2          12  0.492984  14.909048  222.27971  14.917197   
3          3          12 -2.007783  14.349675  205.91318  14.489458   
4          4          12  1.876135  13.039218  170.02121  13.173500   

   AF3_skewness  AF3_kurtosis  AF3_delta_power  AF3_delta_relative_power  ...  \
0     -0.339354     -0.337851       114.806351                  0.573799  ...   
1     -0.468445     -0.308483       185.131073                  0.716253  ...   
2     -0.283593     -0.254668       141.955475                  0.627093  ...   
3     -0.157667     -0.236524        81.054817                  0.508537  ...   
4     -0.063752     -0.338315        59.304790                  0.397247  ...   

   AF4_theta_power  AF4_theta_relative_power  AF4_alpha_power  \
0       323.099518   

In [11]:
# Fixed seed so that all results are reproducible
RANDOM_STATE = 42

# Feature subset sizes to compare (the smallest ones go to the quantum SVM)
CANDIDATE_KS = [10, 20, 50, 100, 150]

# Main metric used to compare the methods
SCORING = 'f1_weighted'

#### Train Test Split

In [12]:
# Separate features, target and subject ids.
# subject_id is NOT a feature, but we keep it as `groups` so that the
# cross-validation never puts the same person in both train and test.
X_ = DATASET_EXT.drop(columns=['sample_id', 'subject_id', 'label'])
y_ = DATASET_EXT['label']
subjects = DATASET_EXT['subject_id']

UNIQUE_SUBJECTS = np.unique(subjects)

SUB_train, SUB_test = train_test_split(
    UNIQUE_SUBJECTS,
    test_size=0.25,
    random_state=RANDOM_STATE
)

print(SUB_train.shape)
print(SUB_test.shape)

# Train matrix
train=np.isin(subjects, SUB_train)
X_train=X_[train]
y_train=y_[train]
subjects_train = subjects[train]

print(X_train.shape)
print(y_train.shape)

# Test matrix
test=np.isin(subjects, SUB_test)
X_test=X_[test]
y_test=y_[test]
subjects_test = subjects[test]

print(X_test.shape)
print(y_test.shape)

# Make sure the data is clean before any modelling
assert X_.isna().sum().sum() == 0, "NaN values found in features"
assert all(np.issubdtype(t, np.number) for t in X_.dtypes), "Non-numeric columns found"

# Grouped, stratified CV: no subject appears in both train and validation folds.
# The same object is used for both methods, so they see identical splits.
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

(33,)
(12,)
(19602, 238)
(19602,)
(7128, 238)
(7128,)


## Quantum

In [13]:
# Fixed seed so that all results are reproducible
RANDOM_STATE = 42

# Feature subset sizes to compare (the smallest ones go to the quantum SVM)
CANDIDATE_KS = [10, 20, 50, 100, 150]

# Main metric used to compare the methods
SCORING = 'f1_weighted'

In [14]:
# Separate features, target and subject ids.
# subject_id is NOT a feature, but we keep it as `groups` so that the
# cross-validation never puts the same person in both train and test.
X_ = DATASET_EXT.drop(columns=['sample_id', 'subject_id', 'label'])
y_ = DATASET_EXT['label']
subjects = DATASET_EXT['subject_id']

UNIQUE_SUBJECTS = np.unique(subjects)

SUB_train, SUB_test = train_test_split(
    UNIQUE_SUBJECTS,
    test_size=0.25,
    random_state=RANDOM_STATE
)

print(SUB_train.shape)
print(SUB_test.shape)

# Train matrix
train=np.isin(subjects, SUB_train)
X_train=X_[train]
y_train=y_[train]
subjects_train = subjects[train]

print(X_train.shape)
print(y_train.shape)

# Test matrix
test=np.isin(subjects, SUB_test)
X_test=X_[test]
y_test=y_[test]
subjects_test = subjects[test]

print(X_test.shape)
print(y_test.shape)

# Make sure the data is clean before any modelling
assert X_.isna().sum().sum() == 0, "NaN values found in features"
assert all(np.issubdtype(t, np.number) for t in X_.dtypes), "Non-numeric columns found"

# Grouped, stratified CV: no subject appears in both train and validation folds.
# The same object is used for both methods, so they see identical splits.
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

(33,)
(12,)
(19602, 238)
(19602,)
(7128, 238)
(7128,)


In [15]:
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import PegasosQSVC
from sklearn.base import BaseEstimator, ClassifierMixin

# Do not remove, mandantory in order to not have key error later in the fit
X_train=X_train.reset_index(drop=True)
y_train=y_train.reset_index(drop=True)
X_test=X_test.reset_index(drop=True)
y_test=y_test.reset_index(drop=True)

In [16]:
Q_CANDIDATE_KS = [2]

Q_reducers = {
    "ANOVA": (
        SelectKBest(score_func=f_classif),
        "reducer__k",
    ),
    "PCA": (
        PCA(),
        "reducer__n_components",
    ),
}

In [17]:
class DimensionQSVC(ClassifierMixin, BaseEstimator):

    def __init__(self, feature_dimension=2, reps=2, C=1.0):
        self.feature_dimension = feature_dimension
        self.reps = reps
        self.C = C

    def fit(self, X, y):
        if self.feature_dimension > X.shape[1]:
            raise ValueError("feature_dimension exceeds input dimensions")

        X_selected = X[:, :self.feature_dimension]

        feature_map = zz_feature_map(
            feature_dimension=self.feature_dimension,
            reps=self.reps,
        )

        quantum_kernel = FidelityQuantumKernel(
            feature_map=feature_map,
        )

        self.model_ = PegasosQSVC(
            quantum_kernel=quantum_kernel,
            C=self.C,
        )

        self.model_.fit(
            np.asarray(X_selected),
            np.asarray(y).ravel()
        )
        self.classes_ = self.model_.classes_
        return self

    def predict(self, X):
        X_selected = X[:, :self.feature_dimension]
        return self.model_.predict(X_selected)

In [18]:
def quantum_workerPCA(n: int):
    t0 = time.time()

    feature_map = zz_feature_map(
        feature_dimension=n,
        reps=2,
    )

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n)),
        ("pegasos", PegasosQSVC(
            quantum_kernel=FidelityQuantumKernel(feature_map=feature_map),
            C=1.0,
            num_steps=100
        ))
    ])

    # Convert DataFrames to NumPy arrays
    X_train_np = np.asarray(X_train)
    X_test_np = np.asarray(X_test)
    y_train_np = np.asarray(y_train).ravel()
    y_test_np = np.asarray(y_test).ravel()

    pipeline.fit(X_train_np, y_train_np)

    # Apply the fitted preprocessing steps
    X_test_scaled = pipeline.named_steps["scaler"].transform(X_test_np)
    X_test_selected = pipeline.named_steps["pca"].transform(X_test_scaled)

    # Convert the transformed features to NumPy explicitly
    X_test_selected = np.asarray(X_test_selected)

    # Predict and evaluate using the transformed test data
    qsvc_fitted = pipeline.named_steps["pegasos"]
    y_pred = qsvc_fitted.predict(X_test_selected)
    score = accuracy_score(y_test_np, y_pred)

    print(y_pred)
    print(f"PCA Done: {time.time() - t0:.2f}s - score: {score:.4f}")

    return pipeline


In [19]:
def quantum_workerANOVA(n: int):
    t0 = time.time()

    feature_map = zz_feature_map(
        feature_dimension=n,
        reps=2,
    )

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("anova", SelectKBest(score_func=f_classif, k=n)),
        ("pegasos", PegasosQSVC(
            quantum_kernel=FidelityQuantumKernel(feature_map=feature_map),
            C=1.0,
            num_steps=100
        ))
    ])

    # Convert DataFrames to NumPy arrays
    X_train_np = np.asarray(X_train)
    X_test_np = np.asarray(X_test)
    y_train_np = np.asarray(y_train).ravel()
    y_test_np = np.asarray(y_test).ravel()

    pipeline.fit(X_train_np, y_train_np)

    # Apply the fitted preprocessing steps
    X_test_scaled = pipeline.named_steps["scaler"].transform(X_test_np)
    X_test_selected = pipeline.named_steps["anova"].transform(X_test_scaled)

    # Convert the transformed features to NumPy explicitly
    X_test_selected = np.asarray(X_test_selected)

    # Predict and evaluate using the transformed test data
    qsvc_fitted = pipeline.named_steps["pegasos"]
    y_pred = qsvc_fitted.predict(X_test_selected)
    score = accuracy_score(y_test_np, y_pred)

    print(y_pred)
    print(f"ANOVA Done: {time.time() - t0:.2f}s - score: {score:.4f}")

    return pipeline


In [20]:
import threading

for k in [2, 3]:
    print(f"Dim: {k}")
    threads = []
    
    threads.append(threading.Thread(target=quantum_workerPCA, args=(k,)))
    threads.append(threading.Thread(target=quantum_workerANOVA, args=(k,)))

    for t in threads:
        t.start()
    
    for t in threads:
        t.join()

Dim: 2
[0 1 0 ... 0 1 0]
PCA Done: 1273.27s - score: 0.4966
[1 1 1 ... 1 1 1]
ANOVA Done: 1282.99s - score: 0.5000
Dim: 3
[0 0 0 ... 0 0 0]
ANOVA Done: 2453.21s - score: 0.5000
[0 0 0 ... 0 0 0]
PCA Done: 2468.25s - score: 0.5000
